# Visual Genome Captioning - Complete Pipeline 🚀

Notebook này chứa toàn bộ quy trình thiết lập, tải dữ liệu, xử lý và huấn luyện cho bài toán **Trích xuất thông tin** và **Sinh mô tả ảnh** trên bộ dữ liệu Visual Genome.

## 0. Cấu hình (Configuration)

In [9]:
# ======= CẤU HÌNH THỰC THI (Sửa tại đây) ======= 

# Chọn mode huấn luyện: 'task1', 'task2', hoặc 'both'
TRAINING_MODE = 'both'

# Tải dữ liệu metadata từ Stanford/HuggingFace (True khi chạy lần đầu)
DOWNLOAD_DATA = True

# Cách tải ảnh gốc: 'none' (không tải), 'sample' (chỉ tải mẫu), 'all' (tải toàn bộ)
IMAGE_DOWNLOAD_MODE = 'sample'
SAMPLE_SIZE = 200

# Có sử dụng cache đặc trưng trước khi train không (Tiết kiệm thời gian RAM/GPU rất nhiều)
PRE_EXTRACT_FEATURES = True

MAX_EPOCHS = 20
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
MAX_SAMPLES = 1000  # Giới hạn số mẫu train (dùng để test luồng code chạy thành công)

## 1. Cài đặt Môi trường (Setup)

In [2]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
%pip install transformers timm datasets tqdm omegaconf scikit-learn pyyaml wandb
%pip install opencv-python pillow matplotlib seaborn

Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import sys
import torch

current_dir = os.path.abspath(os.getcwd())
if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

to_remove = [k for k in sys.modules.keys() if k == 'src' or k.startswith('src.')]
for k in to_remove:
    del sys.modules[k]

print(f"Project Root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {DEVICE}")

Project Root: c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj
PyTorch version: 2.10.0+cpu
Using Device: cpu


## 2. Load Data

In [4]:
from src.data.download import download_and_extract_metadata, download_vg_images, verify_download
import json
if DOWNLOAD_DATA:
    download_and_extract_metadata(keep_zip=False)
    
    if IMAGE_DOWNLOAD_MODE == 'sample':
        with open('data/raw/image_data.json', 'r') as f:
            img_data = json.load(f)
        sample_ids = [img['image_id'] for img in img_data[:SAMPLE_SIZE]]
        
        print(f"Bắt đầu tải bộ sample {len(sample_ids)} ảnh...")
        download_vg_images(sample_ids)
        
    elif IMAGE_DOWNLOAD_MODE == 'all':
        print("Cảnh báo: Tải bộ dữ liệu ảnh rất nặng.")
        
verify_download()


--- Đang xử lý objects ---


Tải objects_v1_2.json.zip: 100%|██████████| 69.9M/69.9M [00:07<00:00, 9.59MiB/s]


[Unzip] Đã giải nén 1 files từ objects_v1_2.json.zip
[Cleanup] Đã xóa objects_v1_2.json.zip
[Skip] attributes.json đã tồn tại.

--- Đang xử lý relationships ---


Tải relationships_v1_2.json.zip: 100%|██████████| 73.5M/73.5M [00:10<00:00, 7.37MiB/s]


[Unzip] Đã giải nén 1 files từ relationships_v1_2.json.zip
[Cleanup] Đã xóa relationships_v1_2.json.zip
[Skip] image_data.json đã tồn tại.
Bắt đầu tải bộ sample 200 ảnh...
[Images] 5 ảnh đã có | 195 ảnh cần tải


[Images] Tải thành công: 195/195 ảnh mới

--- Kiểm tra dữ liệu RAW ---
  ✅  objects.json (412.6 MB)
  ✅  attributes.json (462.6 MB)
  ✅  relationships.json (709.6 MB)
  ✅  image_data.json (17.6 MB)


{'objects.json': True,
 'attributes.json': True,
 'relationships.json': True,
 'image_data.json': True}

## 3. Data Preprocessing (Tạo Vocab & Split)

In [5]:
from src.data.preprocessing import build_vocab_and_splits

if TRAINING_MODE in ['task1', 'both']:
    build_vocab_and_splits(task='task1')
if TRAINING_MODE in ['task2', 'both']:
    build_vocab_and_splits(task='task2')

--- [Task 1] Bắt đầu Preprocessing ---
Đang load objects file...
Đang load attributes file...
Đang đếm tần suất object & attributes...


Parsing: 100%|██████████| 108077/108077 [00:21<00:00, 5093.31it/s] 


Đang mapping dữ liệu sang integer indices...
Tổng hợp được 2071985 samples hợp lệ.
--- Hoàn tất Task 1 Preprocessing ---
--- [Task 2] Bắt đầu Preprocessing ---


Parsing Relationships: 100%|██████████| 108077/108077 [00:21<00:00, 4971.52it/s] 


Đang mapping dữ liệu sang integer indices...
--- Hoàn tất Task 2 Preprocessing ---


## 4. Trích Xuất Đặc Trưng (Feature Extraction)
Chỉ xuất đặc trưng từ những ảnh `.jpg` bạn đã tải về thư mục `images`.

In [6]:
from src.features.feature_extractor import extract_task1_features, extract_task2_features

if PRE_EXTRACT_FEATURES:
    if TRAINING_MODE in ['task1', 'both']:
        print("Extracting features Task 1...")
        extract_task1_features(
            annotation_file="data/processed/task1/train/annotations.json",
            image_dir="data/raw/images",
            output_file="data/processed/task1/features/train_features.pt",
            backbone="resnet50",
            batch_size=32, device=DEVICE
        )
        extract_task1_features(
            annotation_file="data/processed/task1/val/annotations.json",
            image_dir="data/raw/images",
            output_file="data/processed/task1/features/val_features.pt",
            backbone="resnet50",
            batch_size=32, device=DEVICE
        )
        
    if TRAINING_MODE in ['task2', 'both']:
        print("Extracting features Task 2...")
        extract_task2_features(
            annotation_file="data/processed/task2/train/annotations.json",
            image_dir="data/raw/images",
            output_file="data/processed/task2/features/train_features.pt",
            backbone="resnet50",
            batch_size=32, device=DEVICE
        )
        extract_task2_features(
            annotation_file="data/processed/task2/val/annotations.json",
            image_dir="data/raw/images",
            output_file="data/processed/task2/features/val_features.pt",
            backbone="resnet50",
            batch_size=32, device=DEVICE
        )

Extracting features Task 1...
Extracting Task 1 features using resnet50...


Processing images: 100%|██████████| 77093/77093 [01:33<00:00, 825.41it/s]  


Saved 3516 features to data\processed\task1\features\train_features.pt
Extracting Task 1 features using resnet50...


Processing images: 100%|██████████| 14817/14817 [00:00<00:00, 48148.84it/s]


Saved 0 features to data\processed\task1\features\val_features.pt
Extracting features Task 2...
Extracting Task 2 features using resnet50...


Processing images: 100%|██████████| 71504/71504 [02:05<00:00, 568.90it/s]  


Saved 3991 features to data\processed\task2\features\train_features.pt
Extracting Task 2 features using resnet50...


Processing images: 100%|██████████| 17487/17487 [00:00<00:00, 53065.80it/s]


Saved 0 features to data\processed\task2\features\val_features.pt


## 5. Huấn Luyện (Training)

In [10]:
if TRAINING_MODE in ['task1', 'both']:
    print("========== BẮT ĐẦU TRAINING TASK 1 ==========")
    from src.training.task1_trainer import Task1Trainer
    from src.models.task1.object_classifier import ObjectClassifier
    from src.models.task1.attribute_classifier import AttributeClassifier
    from src.data.task1_dataset import build_task1_datasets
    from torch.utils.data import DataLoader
    
    print("Khởi tạo Dataset...")
    train_ds, val_ds, _ = build_task1_datasets(
        processed_dir="data/processed/task1", 
        image_dir="data/raw/images", 
        max_samples=MAX_SAMPLES,
        use_feature_cache=PRE_EXTRACT_FEATURES
    )
    
    # Mô hình
    obj_model = ObjectClassifier(num_classes=train_ds.num_objects, feature_dim=2048, device=DEVICE)
    attr_model = AttributeClassifier(num_attributes=train_ds.num_attributes, feature_dim=2048, device=DEVICE)
    
    obj_opt = torch.optim.Adam(obj_model.parameters(), lr=LEARNING_RATE)
    attr_opt = torch.optim.Adam(attr_model.parameters(), lr=LEARNING_RATE)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    trainer1 = Task1Trainer(device=DEVICE, 
        object_model=obj_model, attribute_model=attr_model,
        train_loader=train_loader, val_loader=val_loader,
        object_optimizer=obj_opt, attribute_optimizer=attr_opt,
        max_epochs=MAX_EPOCHS
    )
    print("Bắt đầu Epochs...")
    trainer1.train()
    print("Hoàn tất Training Task 1.")


if TRAINING_MODE in ['task2', 'both']:
    print("========== BẮT ĐẦU TRAINING TASK 2 ==========")
    from src.training.task2_trainer import Task2Trainer
    from src.models.task2.relation_classifier import RelationClassifier
    from src.data.task2_dataset import build_task2_datasets
    from torch.utils.data import DataLoader
    
    print("Khởi tạo Dataset...")
    train_ds2, val_ds2, _ = build_task2_datasets(
        processed_dir="data/processed/task2", 
        image_dir="data/raw/images", 
        max_samples=MAX_SAMPLES,
        use_feature_cache=PRE_EXTRACT_FEATURES
    )
    
    rel_model = RelationClassifier(num_relations=train_ds2.num_relations, feature_dim=2048, device=DEVICE)
    rel_opt = torch.optim.Adam(rel_model.parameters(), lr=LEARNING_RATE)
    
    train_loader2 = DataLoader(train_ds2, batch_size=BATCH_SIZE, shuffle=True)
    val_loader2 = DataLoader(val_ds2, batch_size=BATCH_SIZE, shuffle=False)
    
    trainer2 = Task2Trainer(device=DEVICE, 
        model=rel_model,
        train_loader=train_loader2, val_loader=val_loader2,
        optimizer=rel_opt, max_epochs=MAX_EPOCHS
    )
    print("Bắt đầu Epochs...")
    trainer2.train()
    print("Hoàn tất Training Task 2.")

========== BẮT ĐẦU TRAINING TASK 1 ==========
Khởi tạo Dataset...
[Task1Dataset] Loaded 1450389 annotations từ data\processed\task1\train\annotations.json
[Task1Dataset] Loading feature cache: data\processed\task1\features\train_features.pt
[Task1Dataset] Loaded 3516 cached features
[Task1Dataset] Giới hạn 1000 samples (debug mode)
ObjectAttributeDataset [train]: 1000 ROIs | 61 images | 151 objects | 101 attributes | cache=✅
[Task1Dataset] Loaded 310798 annotations từ data\processed\task1\val\annotations.json
[Task1Dataset] Loading feature cache: data\processed\task1\features\val_features.pt
[Task1Dataset] Loaded 0 cached features
[Task1Dataset] Giới hạn 1000 samples (debug mode)
ObjectAttributeDataset [val]: 1000 ROIs | 48 images | 151 objects | 101 attributes | cache=✅


2026-04-04 20:14:13 [INFO] TensorBoard logging to: logs\tensorboard\vg_caption
2026-04-04 20:14:13 [INFO] Starting training for 20 epochs
2026-04-04 20:14:13 [INFO] Model: ObjectClassifier
2026-04-04 20:14:13 [INFO] Train samples: 1000
2026-04-04 20:14:13 [INFO] Val samples: 1000
2026-04-04 20:14:13 [INFO] Step 1: batch/batch_loss=5.7153, batch/batch_object_loss=4.9888, batch/batch_attribute_loss=0.6959, batch/batch_idx=0


[Task1Dataset] Loaded 310798 annotations từ data\processed\task1\test\annotations.json
[Task1Dataset] Giới hạn 1000 samples (debug mode)
ObjectAttributeDataset [test]: 1000 ROIs | 45 images | 151 objects | 101 attributes | cache=❌
Bắt đầu Epochs...


2026-04-04 20:14:14 [INFO] Step 51: batch/batch_loss=5.5577, batch/batch_object_loss=4.8503, batch/batch_attribute_loss=0.6964, batch/batch_idx=50
2026-04-04 20:14:14 [INFO] Step 0: epoch/train_loss=5.5868, epoch/lr=0.0001, epoch/object_accuracy=0.0620, epoch/object_precision=0.0005, epoch/object_recall=0.0085, epoch/object_f1=0.0010, epoch/object_mAP=0.0262, epoch/attribute_precision_macro=0.0054, epoch/attribute_recall_macro=0.2725, epoch/attribute_f1_macro=0.0043, epoch/attribute_precision_micro=0.0037, epoch/attribute_recall_micro=0.5824, epoch/attribute_f1_micro=0.0074, epoch/attribute_precision_per_class=[0.0, 0.104, 0.3333333333333333, 0.0, 0.014, 0.0, 0.028, 0.006, 0.007, 0.0, 0.0, 0.006006006006006006, 0.0, 0.0027063599458728013, 0.001, 0.002, 0.004020100502512563, 0.011, 0.0, 0.001, 0.0, 0.001, 0.0, 0.002, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.004, 0.0, 0.0, 0.0, 0.001, 0.003, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.001, 0.0, 0.0, 0.0, 0.0, 0.002, 0.0, 0.0, 0.0, 0.0, 0.001, 

Removed old checkpoint: checkpoints\vg_caption\task1_attribute_epoch_0.pth
Checkpoint saved: checkpoints\vg_caption\task1_object_epoch_0.pth
Removed old checkpoint: checkpoints\vg_caption\task1_object_epoch_1.pth
Checkpoint saved: checkpoints\vg_caption\task1_attribute_epoch_0.pth


...

Removed old checkpoint: checkpoints\vg_caption\task2_epoch_12.pth
Checkpoint saved: checkpoints\vg_caption\task2_epoch_17.pth


2026-04-04 20:14:26 [INFO] Step 18: epoch/train_loss=1.3724, epoch/lr=0.0001, epoch/accuracy=0.3060, epoch/precision=0.0071, epoch/recall=0.0221, epoch/f1=0.0102, epoch/mAP=0.0303, epoch/val_loss=4.7004
2026-04-04 20:14:26 [INFO] Step 1198: batch/batch_loss=1.2670, batch/batch_idx=0
2026-04-04 20:14:27 [INFO] Step 1248: batch/batch_loss=0.9544, batch/batch_idx=50


Removed old checkpoint: checkpoints\vg_caption\task2_epoch_13.pth
Checkpoint saved: checkpoints\vg_caption\task2_epoch_18.pth


2026-04-04 20:14:27 [INFO] Step 19: epoch/train_loss=1.3322, epoch/lr=0.0001, epoch/accuracy=0.3040, epoch/precision=0.0085, epoch/recall=0.0220, epoch/f1=0.0103, epoch/mAP=0.0304, epoch/val_loss=4.6926
2026-04-04 20:14:27 [INFO] Loaded best checkpoint
2026-04-04 20:14:27 [INFO] Training completed


Removed old checkpoint: checkpoints\vg_caption\task2_epoch_14.pth
Checkpoint saved: checkpoints\vg_caption\task2_epoch_19.pth
Checkpoint loaded: checkpoints\vg_caption\task2_epoch_19.pth
Epoch: 19, Loss: 1.3321637265265933
Hoàn tất Training Task 2.


## 6. Sinh Caption & Đánh giá

In [8]:
from src.models.caption.caption_generator import CaptionGenerator

# Ví dụ inference giả định:
# captioner = CaptionGenerator(task1_model=obj_model, task2_model=rel_model, strategy='template')
# sentence = captioner.generate(image_roi)
# print(sentence)